<a href="https://colab.research.google.com/github/Sourabh-IISER/LEAF_CLASSIFICATION/blob/main/notebooks/model%202.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install or verify required libraries in Google Colab
!pip install -q opencv-python-headless scikit-image scikit-learn matplotlib pandas numpy joblib tqdm

import os
import re
import sys
import json
import time
import shutil
import hashlib
import zipfile
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image, UnidentifiedImageError

from skimage.feature import local_binary_pattern
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    train_test_split,
    cross_val_score
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = False
print("All libraries imported successfully!")

In [ ]:
# Mount Google Drive when running inside Google Colab
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    else:
        print("Google Drive is already mounted at /content/drive.")
except ImportError:
    print("Not running inside Google Colab environment; using local filesystem paths.")

# ==============================================================================
# CONFIGURATION SETTINGS (Modify ZIP_PATH and PROJECT_DIR to match your Drive)
# ==============================================================================
ZIP_PATH = Path("/content/drive/MyDrive/Leaf_Classification/Dataset/archive.zip")
PROJECT_DIR = Path("/content/drive/MyDrive/leaf_project")

# Subdirectories inside PROJECT_DIR
SELECTED_IMAGES_DIR = PROJECT_DIR / "selected_images"
UNKNOWN_IMAGES_DIR  = PROJECT_DIR / "unknown_eval_images"
METADATA_DIR        = PROJECT_DIR / "metadata"
FEATURES_DIR        = PROJECT_DIR / "features"
MODELS_DIR          = PROJECT_DIR / "models"
OUTPUTS_DIR         = PROJECT_DIR / "outputs"

# Target Known Species (Canonical Label -> Accepted folder token variations)
TARGET_SPECIES = ["Mango", "Lemon", "Guava", "Jamun", "Pomegranate"]

SPECIES_ALIASES = {
    "Mango":       ["mango", "mangifera", "mangifera_indica", "p0a", "p0b"],
    "Lemon":       ["lemon", "citrus_limon", "nimbu", "p10a", "p10b"],
    "Guava":       ["guava", "gauva", "psidium", "psidium_guajava", "amrood", "p3a", "p3b"],
    "Jamun":       ["jamun", "syzygium", "syzygium_cumini", "java_plum", "jambolan", "p5a", "p5b"],
    "Pomegranate": ["pomegranate", "punica", "punica_granatum", "anar", "p9a", "p9b"]
}

HEALTH_KEYWORDS = {
    "Healthy":  ["healthy", "normal", "fresh", "good", "clean"],
    "Diseased": ["diseased", "disease", "infected", "sick", "damaged", "spot", "blight", "rust", "curl", "rot", "mildew"]
}

# Image & Feature Settings
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
IMAGE_SIZE = (128, 128)  # (Width, Height)
LBP_POINTS = 8
LBP_RADIUS = 1
LBP_METHOD = "uniform"   # Produces P + 2 = 10 histogram bins
HSV_BINS   = (12, 8, 8)  # 12 Hue bins + 8 Saturation bins + 8 Value bins
RGB_BINS   = (8, 8, 8)   # 8 Red + 8 Green + 8 Blue bins

# Dataset Splitting & Model Settings
RANDOM_SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

# Default/Provisional Unknown Rejection Threshold (will be tuned on Validation set)
DEFAULT_CONFIDENCE_THRESHOLD = 0.55

# Maximum number of held-out unknown-species images to extract for evaluation
MAX_UNKNOWN_SPECIES_IMAGES_PER_FOLDER = 60

# Create all persistent project directories automatically
for directory in [
    PROJECT_DIR, SELECTED_IMAGES_DIR, UNKNOWN_IMAGES_DIR,
    METADATA_DIR, FEATURES_DIR, MODELS_DIR, OUTPUTS_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)
print(f"Project directory structure ready at: {PROJECT_DIR}")

# Verify whether the ZIP archive exists
if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"\n[ERROR] Dataset ZIP not found at: {ZIP_PATH}\n"
        f"Please check that:\n"
        f"  1. Google Drive is mounted properly.\n"
        f"  2. The filename and path in ZIP_PATH match the exact location in your Google Drive.\n"
        f"  Example: /content/drive/MyDrive/leaf_dataset.zip"
    )
else:
    zip_size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
    print(f"Found ZIP archive: {ZIP_PATH} ({zip_size_mb:.2f} MB)")

In [ ]:
def inspect_zip_archive(zip_path: Path, supported_exts: set):
    """
    Inspects the ZIP archive header without extracting files.
    Returns folder image counts, unsupported file counts, and archive entry metadata.
    """
    if not zipfile.is_zipfile(zip_path):
        raise ValueError(f"The file at {zip_path} is not a valid ZIP archive.")

    folder_image_counts = Counter()
    folder_unsupported_counts = Counter()
    unsupported_extensions = Counter()
    all_image_entries = []
    filename_occurrences = defaultdict(list)

    with zipfile.ZipFile(zip_path, "r") as zf:
        info_list = zf.infolist()
        for info in tqdm(info_list, desc="Scanning ZIP entries"):
            # Skip directory entries and macOS/Windows metadata clutter
            norm_name = info.filename.replace("\\", "/").strip("/")
            if info.is_dir() or not norm_name:
                continue
            parts = norm_name.split("/")
            if any(p.startswith("__MACOSX") or p.startswith(".") for p in parts):
                continue

            ext = Path(norm_name).suffix.lower()
            parent_folder = "/".join(parts[:-1]) if len(parts) > 1 else "<ROOT>"
            base_name = parts[-1]

            if ext in supported_exts:
                folder_image_counts[parent_folder] += 1
                filename_occurrences[base_name.lower()].append(norm_name)
                all_image_entries.append({
                    "archive_path": norm_name,
                    "parent_folder": parent_folder,
                    "filename": base_name,
                    "extension": ext,
                    "file_size_bytes": info.file_size,
                    "crc": info.CRC
                })
            else:
                folder_unsupported_counts[parent_folder] += 1
                unsupported_extensions[ext if ext else "<NO_EXT>"] += 1

    duplicate_filenames = {
        fname: paths for fname, paths in filename_occurrences.items() if len(paths) > 1
    }

    return {
        "folder_image_counts": folder_image_counts,
        "folder_unsupported_counts": folder_unsupported_counts,
        "unsupported_extensions": unsupported_extensions,
        "all_image_entries": all_image_entries,
        "duplicate_filenames": duplicate_filenames
    }

zip_inspection = inspect_zip_archive(ZIP_PATH, SUPPORTED_EXTENSIONS)
print(f"Total supported images found in ZIP: {len(zip_inspection['all_image_entries'])}")
print(f"Total distinct image folders in ZIP: {len(zip_inspection['folder_image_counts'])}")
print(f"Duplicate base filenames across archive: {len(zip_inspection['duplicate_filenames'])}")
if zip_inspection["unsupported_extensions"]:
    print("Unsupported extensions skipped:", dict(zip_inspection["unsupported_extensions"]))

In [ ]:
def tokenize_folder_path(folder_path: str):
    """Splits a folder path into normalized lowercase alphanumeric tokens and path segments."""
    segments = [seg.strip().lower() for seg in folder_path.replace("\\", "/").split("/") if seg.strip()]
    tokens = []
    for seg in segments:
        tokens.extend([t for t in re.split(r"[^a-z0-9]+", seg) if t])
    return segments, tokens

def classify_folder_path(folder_path: str, species_aliases: dict, health_keywords: dict):
    """
    Determines the canonical species and health status from a folder path.
    Uses directory segments/tokens only (never individual image filenames).
    Returns: (status, assigned_species, health_condition, matched_species_list, notes)
      where status is one of: 'TARGET', 'AMBIGUOUS', 'NON_TARGET'
    """
    if folder_path == "<ROOT>":
        return "NON_TARGET", None, "Unknown", [], "Files in archive root directory"

    segments, tokens = tokenize_folder_path(folder_path)
    token_set = set(tokens)
    joined_segments = " / ".join(segments)

    matched_species = []
    for canonical_species, aliases in species_aliases.items():
        for alias in aliases:
            alias_tokens = [t for t in re.split(r"[^a-z0-9]+", alias.lower()) if t]
            # Check if all tokens of the alias appear in the folder tokens
            if alias_tokens and all(at in token_set for at in alias_tokens):
                matched_species.append(canonical_species)
                break

    # Infer health status from folder tokens
    matched_health = []
    for h_label, h_words in health_keywords.items():
        if any(hw in token_set for hw in h_words):
            matched_health.append(h_label)

    if len(matched_health) == 1:
        health_condition = matched_health[0]
    elif len(matched_health) > 1:
        health_condition = "Mixed/Ambiguous_Health"
    else:
        health_condition = "Unspecified"

    if len(matched_species) == 1:
        return "TARGET", matched_species[0], health_condition, matched_species, "Matched cleanly to target species"
    elif len(matched_species) > 1:
        return "AMBIGUOUS", None, health_condition, matched_species, f"Multiple species matched: {matched_species}"
    else:
        return "NON_TARGET", None, health_condition, [], "Non-target species or other folder"

# Build folder summary table
folder_records = []
for folder_path, img_count in sorted(zip_inspection["folder_image_counts"].items()):
    status, species, health, matched_list, notes = classify_folder_path(
        folder_path, SPECIES_ALIASES, HEALTH_KEYWORDS
    )
    unsupported_cnt = zip_inspection["folder_unsupported_counts"].get(folder_path, 0)
    folder_records.append({
        "original_folder": folder_path,
        "status": status,
        "assigned_species": species if species else ("AMBIGUOUS" if status == "AMBIGUOUS" else "Excluded (Non-Target)"),
        "health_condition": health,
        "image_count": img_count,
        "unsupported_files": unsupported_cnt,
        "notes": notes
    })

folder_summary_df = pd.DataFrame(folder_records)
print("=== ZIP Folder Inspection & Mapping Summary ===")
display(folder_summary_df)

# Save folder inspection table to metadata directory
folder_summary_csv = METADATA_DIR / "zip_folder_inspection_summary.csv"
folder_summary_df.to_csv(folder_summary_csv, index=False)
print(f"Saved folder summary table to: {folder_summary_csv}")

# Check for ambiguous folders and stop for clarification if any exist
ambiguous_folders = folder_summary_df[folder_summary_df["status"] == "AMBIGUOUS"]
if not ambiguous_folders.empty:
    display(ambiguous_folders)
    raise RuntimeError(
        "STOPPING: Ambiguous folder names were detected that match more than one target species. "
        "Please review the table above and clarify or disambiguate the folder mapping before proceeding."
    )

# Summarize counts per target species
target_folders_df = folder_summary_df[folder_summary_df["status"] == "TARGET"]
species_totals = target_folders_df.groupby("assigned_species")["image_count"].sum().reindex(TARGET_SPECIES, fill_value=0)
print("\n=== Image Counts per Target Known Species (Healthy + Diseased Combined) ===")
for sp, cnt in species_totals.items():
    print(f"  - {sp:12s}: {cnt:6d} images")

missing_species = [sp for sp, cnt in species_totals.items() if cnt == 0]
if missing_species:
    raise RuntimeError(
        f"STOPPING: The following target species have 0 detected images in the ZIP: {missing_species}. "
        f"Please check the folder names in the summary table above or update SPECIES_ALIASES."
    )